# English → Russian: fast scratch Transformer on Colab T4

Stage 1 only. Select **Runtime → Change runtime type → T4 GPU**. The default fast experiment targets **2–3 hours including final evaluation**; this is a target, not a measured T4 guarantee. Run one real epoch first, inspect the projected total, then resume the same checkpoint. CPU measurements and assumptions are documented in `docs/performance_report.md`. The raw data is tracked in Git. No Kaggle API, Helsinki fine-tuning, or final combined notebook is included.

## 1. Clone or update the implementation branch
The default ref is `feat/scratch-transformer`. A clean existing checkout is updated with a fast-forward merge so rerunning this notebook picks up the speed fixes. A dirty checkout is preserved and raises an error; save your changes before updating. Change the ref to `main` after merge if desired. All long-running commands below stream merged stdout/stderr into the cell and propagate failures.

In [ ]:
from pathlib import Path
import subprocess
import sys

def stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

REPO_URL = "https://github.com/vcdx1111/Machine-translation-English-to-Russian-.git"
REPO_REF = "feat/scratch-transformer"
REPO = Path("/content/Machine-translation-English-to-Russian-")
if not REPO.exists():
    stream(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO)])
else:
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Preserve local changes before updating"
    stream(["git", "-C", str(REPO), "fetch", "origin", REPO_REF])
    stream(["git", "-C", str(REPO), "switch", REPO_REF])
    stream(["git", "-C", str(REPO), "merge", "--ff-only", "origin/" + REPO_REF])
%cd /content/Machine-translation-English-to-Russian-
stream(["git", "log", "-1", "--oneline"])

## 2. Install pinned packages
Install into the notebook kernel environment. If Colab asks for a runtime restart, restart and rerun from the top. The command fails immediately if installation fails.

In [ ]:
stream([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
import torch
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime for the fast T4 configuration"
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

## 3. Mount Drive and choose a NEW fast experiment
The smaller model, new tokenizer, shorter cap and batching schedule are incompatible with the old base checkpoints. Use a **new run name**, not the old 512-dimensional run. For this fast experiment's later reconnects, keep the same name and set `RESUME=True`. Epoch checkpoints and results live on Drive. Never run two writers to one run directory. If FP16 is working, startup logs will show `amp_enabled=true`, then the first train/val steps must show `autocast=True, logits=torch.float16`. Check `cuda_peak_gb` after epoch one; memory allocation alone is not GPU utilization.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CONFIG = "configs/transformer_colab_fast.yaml"
RUN_NAME = "scratch_fast_v2_seed42"
RESUME = False
RUN = Path("/content/drive/MyDrive/en_ru_translation") / RUN_NAME
CHECKPOINTS = RUN / "checkpoints"
RESULTS = RUN / "results"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
if RESUME:
    assert (CHECKPOINTS / "last.pt").exists(), "No fast checkpoint found"
else:
    assert not (CHECKPOINTS / "last.pt").exists(), "Use a new run name or set RESUME=True"

## 4. Load and audit the checked-in raw TSV
Read all three columns and create source-grouped train/validation/test splits. The same normalization and split seed are used during training. SentencePiece sees train pairs only. Inspect the measured report before training.

In [ ]:
from src.data import prepare, read_raw
RAW = REPO / "data/raw/rus.txt"
assert RAW.is_file(), "Expected the tracked data/raw/rus.txt"
rows, widths, malformed = read_raw(RAW)
print("Rows:", len(rows), "Columns:", widths, "Malformed:", malformed)
print(rows[:2])
prepare(RAW, "data/processed/full", seed=42)
import json
print(json.dumps(json.loads(Path("data/processed/full/audit.json").read_text()), ensure_ascii=False, indent=2))
del rows

## 5. Measure one real epoch (no test evaluation)
This runs epoch one with the **unchanged planned 12-epoch config**, saves `last.pt`, and stops before test decoding. Progress is streamed in real time: step/total, step ETA, loss, target tokens/sec, AMP status, epoch time and remaining training ETA. Validation computes full loss plus greedy translations on 128 fixed validation sources; no beam search occurs here. Checkpoints include optimizer/scaler/RNG state, so the next training cell continues the same run. Reconnecting runs skip this cell.

In [ ]:
def training_command():
    return [sys.executable, "-u", "-m", "src.train", "--config", CONFIG,
            "--checkpoint-dir", str(CHECKPOINTS), "--results-dir", str(RESULTS)]

if not RESUME:
    stream(training_command() + ["--stop-after-epochs", "1", "--skip-test"])
else:
    print("Resuming: keep existing epoch measurement.")

## 6. Print measured total-time projection
The estimate uses the first/recent real T4 epoch including validation, greedy decoding and observed checkpoint time. Test cost extrapolates validation greedy speed to all test sources plus the 1,024-source beam subset, assuming beam cost scales by width 3. Beam timing is an assumption until final evaluation. Startup/preprocessing, changed generated lengths and Drive delays can alter the total. If the projection exceeds 3 hours, the script prints a warning instead of silently dropping data or epochs. Reduce model/budget in a **new experiment** if needed; old base weights cannot resume into this model.

In [ ]:
import json
estimate = json.loads((RESULTS / "runtime_estimate.json").read_text())
print(json.dumps(estimate, indent=2))
print(f"Measured epoch: {estimate['measured_epoch_seconds']/60:.2f} min")
print(f"Projected training + test: {estimate['projected_total_hours']:.2f} hours (estimate)")
print(f"Remaining training: {estimate['remaining_train_seconds']/60:.1f} min")
if not estimate['within_3_hour_target']:
    print("WARNING: measured run is outside the 3-hour target; do not treat the target as a benchmark.")

## 7. Continue training with live logs, then evaluate
Resume the checkpoint from the measured epoch; no epoch is thrown away or repeated. The fast config uses a 256-dimensional, three-layer model; 8k train-only BPE; 24-token cap; 16,384 padded source+target tokens per batch; FP16; two workers and pinned memory. Training has a 12-epoch ceiling and patience 3. Final test metrics use batched greedy on the whole test split. Beam width 3 runs only on a fixed 1,024-source test subset, with separate greedy/beam subset scores. These are not mixed into one misleading corpus score. Samples contain full-test greedy output. If OOM occurs, use a copied config with fewer `max_tokens` under a new run name.

In [ ]:
stream(training_command() + ["--resume", str(CHECKPOINTS / "last.pt")])

## 8. Standalone evaluation (optional rerun)
Only rerun if final evaluation was interrupted. The checkpoint contains the best validation-loss weights and tokenizer. Metrics label full greedy and the deterministic beam subset separately. For full-test beam, pass `--beam-max-sources 1000000` and expect extra time. Keep different evaluation policies in different output directories.

In [ ]:
RERUN_EVALUATION = False
if RERUN_EVALUATION:
    stream([sys.executable, "-u", "-m", "src.evaluate", "--checkpoint", str(CHECKPOINTS / "last.pt"),
            "--results-dir", str(RESULTS)])

## 9. Inspect and share results
Send metrics, train log, samples, `runtime_estimate.json`, exact YAML/Git commit, GPU name, AMP dtype, epoch time, target tokens/sec and peak memory. Keep weights on Drive. Review both whole-test greedy and same-subset greedy/beam scores before the next stage.

In [ ]:
import csv
metrics = json.loads((RESULTS / "metrics.json").read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print((RESULTS / "train_log.csv").read_text())
print((RESULTS / "samples.txt").read_text())
assert len(list(csv.DictReader((RESULTS / "train_log.csv").open()))) > 0
assert (RESULTS / "samples.txt").read_text().count("EN: ") == 30
print("Results saved in:", RESULTS)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)